In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers crewai crewai-tools deepeval
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = splitter.create_documents([text])

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

In [ ]:
from crewai_tools import tool

@tool
def retrieve_docs(query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    return "\n".join([doc.page_content for doc in docs])

In [ ]:
from crewai import Agent

rag_agent = Agent(
    role="RAG Answer Generator",
    goal="Answer questions using retrieved context",
    backstory="Expert in gene editing who answers strictly from provided context.",
    tools=[retrieve_docs],
    verbose=True
)

In [ ]:
from crewai import Task

rag_task = Task(
    description="""
    Answer the question using retrieved context.

    Question: {question}

    Output format:
    Answer:
    Retrieved Context:
    """,
    agent=rag_agent
)

In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

@tool
def evaluate_answer(question: str, answer: str, context: str) -> str:
    faithfulness = FaithfulnessMetric(threshold=0.7)
    relevancy = AnswerRelevancyMetric(threshold=0.7)

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context
    )

    faithfulness.measure(test_case)
    relevancy.measure(test_case)

    result = {
        "faithfulness": faithfulness.score,
        "relevancy": relevancy.score,
        "verdict": "PASS" if faithfulness.score >= 0.7 and relevancy.score >= 0.7 else "FAIL",
        "reasons": [
            faithfulness.reason,
            relevancy.reason
        ]
    }

    return str(result)

In [ ]:
evaluator_agent = Agent(
    role="Answer Evaluator",
    goal="Evaluate answer quality using metrics",
    backstory="Strict evaluator that checks factual grounding and relevance.",
    tools=[evaluate_answer],
    verbose=True
)

In [ ]:
eval_task = Task(
    description="""
    Evaluate the RAG answer.

    Input:
    Question: {question}
    Answer: {answer}
    Context: {context}

    Output:
    JSON with faithfulness, relevancy, verdict, reasons
    """,
    agent=evaluator_agent
)

In [ ]:
revisor_agent = Agent(
    role="Answer Revisor",
    goal="Fix incorrect or irrelevant answers",
    backstory="Improves answers using evaluator feedback and context.",
    verbose=True
)

In [ ]:
revisor_task = Task(
    description="""
    Revise the answer if it failed evaluation.

    Question: {question}
    Original Answer: {answer}
    Context: {context}
    Evaluation Feedback: {evaluation}

    Instructions:
    - Fix factual errors
    - Improve relevance
    - Use ONLY the provided context

    Output:
    Revised Answer:
    """,
    agent=revisor_agent
)

In [ ]:
from crewai import Crew

crew = Crew(
    agents=[rag_agent, evaluator_agent, revisor_agent],
    tasks=[rag_task, eval_task, revisor_task],
    verbose=True
)

In [ ]:
def run_pipeline(question):
    rag_output = rag_task.execute({"question": question})

    answer = extract_answer(rag_output)
    context = extract_context(rag_output)

    eval_output = eval_task.execute({
        "question": question,
        "answer": answer,
        "context": context
    })

    if "FAIL" in eval_output:
        revised = revisor_task.execute({
            "question": question,
            "answer": answer,
            "context": context,
            "evaluation": eval_output
        })
        return revised

    return answer